# ATC Multi-Agent GRPO Training — Jupyter Server

Mirrors `alurm_runner.sbatch` exactly. Same pinned deps, same training call, same plot output.

**Run order:** top to bottom. Edit the **Config** cell first.

## 0. Config — edit before running

In [ ]:
from pathlib import Path
import os

# ── Paths ──────────────────────────────────────────────────────────────────
# Set REPO_DIR to wherever you cloned the repo on this server.
REPO_DIR   = Path(".").resolve()          # assumes notebook lives inside repo
OUTPUT_DIR = Path("/tmp/atc/outputs")     # change to a persistent path if needed
LOGS_DIR   = Path("/tmp/atc/logs")

# ── Model & training ───────────────────────────────────────────────────────
SMOKE_MODEL   = "Qwen/Qwen2.5-1.5B-Instruct"   # fast sanity-check model
TRAIN_MODEL   = "Qwen/Qwen2.5-7B-Instruct"     # full training model
EPISODES      = 200         # target full run
N_GENERATIONS = 4
SEED          = 42
SKIP_SMOKE    = False       # set True to skip the 1-episode sanity run

# A100-friendly throughput defaults (still stable for GRPO)
BATCH_SIZE      = 8
GRAD_ACCUM      = 1
MAX_NEW_TOKENS  = 384
TEMPERATURE     = 0.7
LOGGING_STEPS   = 1         # print live logs every optimizer step
EVAL_EPISODES   = 20
STRICT_GATES    = True      # fail early when parse/variance/quality gates fail

# ── W&B (optional) ─────────────────────────────────────────────────────────
# Leave blank to run offline. Or set key directly here.
WANDB_KEY    = ""         # e.g. "abcdef123..."  (overrides .env)
WANDB_PROJECT = "atc-multiagent-grpo"

# ── Derived ────────────────────────────────────────────────────────────────
SMOKE_OUTPUT_DIR = OUTPUT_DIR / "atc-smoke"
TRAIN_OUTPUT_DIR = OUTPUT_DIR / "atc-multiagent"
PLOTS_DIR        = OUTPUT_DIR / "plots"

for d in (OUTPUT_DIR, LOGS_DIR, SMOKE_OUTPUT_DIR, TRAIN_OUTPUT_DIR, PLOTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"REPO_DIR  : {REPO_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"PLOTS_DIR : {PLOTS_DIR}")

## 1. Environment variables

In [ ]:
import os, socket, time

os.environ["MASTER_ADDR"]                = "127.0.0.1"
os.environ["MASTER_PORT"]                = str(30000 + (os.getpid() % 2000))
os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
os.environ["PIP_NO_CACHE_DIR"]           = "1"
os.environ["NCCL_DEBUG"]                 = "INFO"
os.environ["NCCL_IB_DISABLE"]            = "1"
os.environ["NCCL_P2P_DISABLE"]           = "0"
os.environ["OMP_NUM_THREADS"]            = "4"
os.environ["MKL_NUM_THREADS"]            = "4"
os.environ["PYTHONUNBUFFERED"]           = "1"
# CRITICAL: disable torch.compile to avoid Dynamo errors with GRPO
os.environ["TORCH_COMPILE_DISABLE"]      = "1"
os.environ["WORLD_SIZE"]                 = "1"
os.environ["RANK"]                       = "0"
os.environ["LOCAL_RANK"]                 = "0"
os.environ["LOCAL_WORLD_SIZE"]           = "1"
os.environ["PYTHONPATH"]                 = str(REPO_DIR) + ":" + os.environ.get("PYTHONPATH", "")

print(f"Hostname       : {socket.gethostname()}")
print(f"Start time     : {time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"MASTER_PORT    : {os.environ['MASTER_PORT']}")
print(f"TORCH_COMPILE_DISABLE=1  (Dynamo disabled)")

## 2. GPU check

In [ ]:
import subprocess, sys

subprocess.run(["nvidia-smi"], check=False)
print(f"Python: {sys.version}")

## 3. Load W&B key from .env (if present)

In [ ]:
import re

def _load_dotenv(path):
    """Minimal .env loader — no external deps needed."""
    p = Path(path)
    if not p.exists():
        return
    for line in p.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        m = re.match(r'^(?:export\s+)?([A-Za-z_][A-Za-z0-9_]*)\s*=\s*(.*)$', line)
        if m:
            k, v = m.group(1), m.group(2).strip().strip('"\'')
            if k not in os.environ:   # don't overwrite already-set vars
                os.environ[k] = v

for env_file in (REPO_DIR / ".env", REPO_DIR / ".env.wandb"):
    _load_dotenv(env_file)
    if env_file.exists():
        print(f"Loaded: {env_file}")

# Explicit key in config cell takes priority
if WANDB_KEY.strip():
    os.environ["WANDB_API_KEY"] = WANDB_KEY.strip()

# Normalize key: strip assignment prefix + whitespace (common .env mistake)
raw_key = os.environ.get("WANDB_API_KEY", "")
raw_key = raw_key.lstrip("\r").split("=")[-1].strip()
if raw_key:
    os.environ["WANDB_API_KEY"]  = raw_key
    os.environ["WANDB_MODE"]     = "online"
    os.environ["WANDB_PROJECT"]  = WANDB_PROJECT
    print(f"W&B key found (len={len(raw_key)}) → mode=online")
else:
    os.environ.pop("WANDB_API_KEY", None)
    os.environ["WANDB_MODE"] = "offline"
    print("W&B offline (no key)")

## 4. Clear stale Unsloth pycache

In [ ]:
import shutil

removed = 0
for p in REPO_DIR.rglob("__pycache__"):
    try:
        shutil.rmtree(p)
        removed += 1
    except Exception:
        pass
print(f"Cleared {removed} __pycache__ dirs")

## 5. Install constraint-compatible packages

This cell follows package metadata constraints for the **locked preinstalled runtime**.
It avoids pins that conflict with `transformers==5.5.4` (notably `huggingface-hub==0.36.2`).

In [ ]:
import subprocess, sys
from importlib.metadata import version, PackageNotFoundError
from packaging.version import Version


def pip(*args):
    cmd = [sys.executable, "-m", "pip"] + list(args)
    print("+", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=False)
    if result.returncode != 0:
        raise RuntimeError(f"pip failed: {' '.join(args[:4])}")


def v(pkg, default="(not installed)"):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return default


print("Detected before install:")
for pkg in [
    "torch", "transformers", "accelerate", "peft", "bitsandbytes", "xformers",
    "trl", "huggingface-hub", "datasets", "multiprocess", "tokenizers",
]:
    print(f"  {pkg:<16} {v(pkg)}")

tf_ver = Version(v("transformers", "0"))

print("\nInstalling vllm for TRL GRPO import-time dependency...")
pip("install", "--upgrade", "--no-input", "vllm==0.19.1")

# Core fix for your traceback: transformers 5.5.4 requires huggingface-hub >= 1.5.0,<2.0
print("\nInstalling runtime-compatible utility deps...")
pip(
    "install", "--upgrade", "--no-input",
    "huggingface-hub>=1.5.0,<2.0",
    "hf_transfer==0.1.9",
    "datasets==4.8.4",
    "multiprocess==0.70.19",
    "xxhash==3.6.0",
    "tyro==0.9.17",
    "wandb==0.19.11",
)

# Keep TRL explicit for notebook behavior consistency.
pip("install", "--upgrade", "--no-input", "trl==0.16.0")

# Unsloth 2026.4.8 metadata caps transformers at <= 5.5.0.
# With the locked transformers==5.5.4 runtime, the published wheel is not metadata-compatible.
print(
    "\nUnsloth install is blocked by locked transformers="
    + str(tf_ver)
    + "; published unsloth 2026.4.8 metadata requires <= 5.5.0."
)

print("\nDone.")

## 6. Verify installations

In [ ]:
import sys
from importlib.metadata import version, PackageNotFoundError
from packaging.version import Version


def v(pkg, default="(not installed)"):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return default


# Force-reimport after pip install in same process
for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ("unsloth", "trl", "peft", "accelerate", "bitsandbytes", "transformers", "huggingface_hub", "vllm")):
        del sys.modules[mod]

import torch
import transformers
import huggingface_hub
import vllm
import trl

print(f"PyTorch        : {torch.__version__}")
print(f"Transformers   : {transformers.__version__}")
print(f"vLLM           : {vllm.__version__}")
print(f"TRL            : {trl.__version__}")
print(f"HF Hub         : {huggingface_hub.__version__}")
print(f"Datasets       : {v('datasets')}")
print(f"Multiprocess   : {v('multiprocess')}")
print(f"CUDA           : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

tf_ver = Version(transformers.__version__)
try:
    import unsloth
    from unsloth import FastLanguageModel
    print(f"Unsloth        : {unsloth.__version__}")
except Exception as exc:
    print(
        "Unsloth import failed under the locked runtime (transformers="
        f"{tf_ver}). Published unsloth 2026.4.8 metadata requires transformers<=5.5.0. "
        f"Error: {exc}"
    )

from trl import GRPOConfig, GRPOTrainer
print("All core imports OK")

## 7. W&B login

In [ ]:
if os.environ.get("WANDB_MODE") == "online":
    try:
        import wandb
        wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True)
        print(f"W&B logged in  project={WANDB_PROJECT}")
    except Exception as exc:
        print(f"W&B login failed ({exc}) → switching to offline")
        os.environ["WANDB_MODE"] = "offline"
else:
    print("W&B offline")

## 8. Smoke gun — 1-episode sanity run

Fast check that the full stack works before committing to a long run.  
Set `SKIP_SMOKE = True` in the Config cell to skip.

In [ ]:
if not SKIP_SMOKE:
    print(f"===== SMOKE GUN START (model={SMOKE_MODEL}, episodes=1) =====")
    result = subprocess.run(
        [
            sys.executable, str(REPO_DIR / "train_grpo.py"),
            "--model",        SMOKE_MODEL,
            "--output_dir",   str(SMOKE_OUTPUT_DIR),
            "--episodes",     "1",
            "--n_generations","2",
            "--seed",         str(SEED),
            "--no_eval",
        ],
        env=os.environ,
        cwd=str(REPO_DIR),
    )
    if result.returncode != 0:
        raise RuntimeError(f"Smoke gun FAILED (exit {result.returncode}) — fix before full run")
    print("===== SMOKE GUN COMPLETE =====")
else:
    print("Smoke gun skipped (SKIP_SMOKE=True)")

## 9. Main training run

In [ ]:
import time

print(f"===== START TRAINING =====")
print(f"Model         : {TRAIN_MODEL}")
print(f"Episodes      : {EPISODES}")
print(f"Generations   : {N_GENERATIONS}")
print(f"Batch/Accum   : {BATCH_SIZE}/{GRAD_ACCUM}")
print(f"Max new tokens: {MAX_NEW_TOKENS}")
print(f"Temperature   : {TEMPERATURE}")
print(f"Logging steps : {LOGGING_STEPS}")
print(f"Eval episodes : {EVAL_EPISODES}")
print(f"Strict gates  : {STRICT_GATES}")
print(f"Output        : {TRAIN_OUTPUT_DIR}")
print()

train_env = dict(os.environ)
train_env["ATC_STRICT_GATES"] = "1" if STRICT_GATES else "0"

t0 = time.monotonic()
result = subprocess.run(
    [
        sys.executable, str(REPO_DIR / "train_grpo.py"),
        "--model",           TRAIN_MODEL,
        "--output_dir",      str(TRAIN_OUTPUT_DIR),
        "--episodes",        str(EPISODES),
        "--n_generations",   str(N_GENERATIONS),
        "--batch_size",      str(BATCH_SIZE),
        "--grad_accum",      str(GRAD_ACCUM),
        "--max_new_tokens",  str(MAX_NEW_TOKENS),
        "--temperature",     str(TEMPERATURE),
        "--logging_steps",   str(LOGGING_STEPS),
        "--eval_episodes",   str(EVAL_EPISODES),
        "--seed",            str(SEED),
    ],
    env=train_env,
    cwd=str(REPO_DIR),
)
elapsed = time.monotonic() - t0

if result.returncode != 0:
    raise RuntimeError(f"Training FAILED (exit {result.returncode})")
print(f"===== TRAINING COMPLETE ({elapsed/60:.1f} min) =====")

## 10. Generate plots

In [ ]:
import json
import sys

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import matplotlib
matplotlib.use("Agg")
from training.plot_rewards import plot_training_curves, plot_eval_comparison

generated = []

# ── Training curves ────────────────────────────────────────────────────────
curves_path = TRAIN_OUTPUT_DIR / "reward_curves.json"
if curves_path.exists():
    data = json.loads(curves_path.read_text())
    plot_training_curves(data, save_dir=str(PLOTS_DIR), show=False)
    generated.append(PLOTS_DIR / "training_curves.png")
    print(f"Saved: training_curves.png")
else:
    print(f"[WARN] {curves_path} not found")

# ── Before / after comparison ──────────────────────────────────────────────
base_path    = TRAIN_OUTPUT_DIR / "base_model_metrics.json"
trained_path = TRAIN_OUTPUT_DIR / "trained_model_metrics.json"
if base_path.exists() and trained_path.exists():
    eval_data = {
        "base":    json.loads(base_path.read_text()),
        "trained": json.loads(trained_path.read_text()),
    }
    plot_eval_comparison(eval_data, save_dir=str(PLOTS_DIR), show=False)
    generated.append(PLOTS_DIR / "eval_comparison.png")
    print(f"Saved: eval_comparison.png")
else:
    print(f"[WARN] base/trained metrics not found — skipping eval_comparison")

print(f"\nAll plots → {PLOTS_DIR}")

## 11. Display plots

In [ ]:
from IPython.display import Image, display

for png in sorted(PLOTS_DIR.glob("*.png")):
    print(f"── {png.name} ──")
    display(Image(filename=str(png), width=900))

## 12. Output file summary

In [ ]:
print(f"Output directory: {TRAIN_OUTPUT_DIR}")
print()
for f in sorted(TRAIN_OUTPUT_DIR.rglob("*")):
    if f.is_file():
        size = f.stat().st_size
        unit = "KB" if size < 1_000_000 else "MB"
        val  = size / 1_000 if size < 1_000_000 else size / 1_000_000
        print(f"  {f.relative_to(TRAIN_OUTPUT_DIR):<50}  {val:6.1f} {unit}")

print()
print(f"Plots directory: {PLOTS_DIR}")
for f in sorted(PLOTS_DIR.glob("*.png")):
    print(f"  {f.name}")